## setting up

In [1]:
#package
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [4]:
#environmental info
import sys
import pandas as pd
import numpy as np
import matplotlib
import statsmodels

# Print package versions
print(f"Python:       {sys.version.split()[0]}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"statsmodels:  {statsmodels.__version__}")

Python:       3.13.16
pandas:       2.2.3
numpy:        2.1.3
matplotlib:   3.10.0
statsmodels:  0.15.0


In [5]:
#set current directrory
#os.chdir("")

In [6]:
#read data
#※data ignored as a public repository
df = pd.read_stata("data/INJURY.DTA")

In [7]:
df.columns

Index(['durat', 'afchnge', 'highearn', 'male', 'married', 'hosp', 'indust',
       'injtype', 'age', 'prewage', 'totmed', 'injdes', 'benefit', 'ky', 'mi',
       'ldurat', 'afhigh', 'lprewage', 'lage', 'ltotmed', 'head', 'neck',
       'upextr', 'trunk', 'lowback', 'lowextr', 'occdis', 'manuf', 'construc',
       'highlpre'],
      dtype='object')

In [8]:
#check ky
df['ky'].describe()

,ky
count,7150.000000
mean,0.786853
std,0.409559
min,0.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,1.000000


In [9]:
#restrict ky ==1
df = df[df['ky'] == 1].copy()

In [10]:
# check filtered
print(df['ky'].value_counts())

ky
1    5626
Name: count, dtype: int64


In [11]:
print(df['ldurat'].describe())

count    5626.000000
mean        1.277745
std         1.282157
min        -1.386294
25%         0.693147
50%         1.386294
75%         2.079442
max         5.204007
Name: ldurat, dtype: float64


In [12]:
vars_used = [
    'ldurat', 'highearn', 'afchnge',
    'lage', 'male', 'married', 'hosp',
    'indust', 'injtype', 'injdes'
]

# Drop observations with missing values
df = df.dropna(subset=vars_used).copy()

print("Observations:", len(df))
print(df[vars_used].isna().sum())

Observations: 5347
ldurat      0
highearn    0
afchnge     0
lage        0
male        0
married     0
hosp        0
indust      0
injtype     0
injdes      0
dtype: int64


## A1 Simple difference-in-differences.

In [13]:
# OLS（Non-robust standard errors）
model1 = smf.ols(
    formula='ldurat ~ highearn * afchnge',
    data=df
).fit(cov_type='nonrobust')

print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                 ldurat   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     39.58
Date:                Mon, 26 Oct 2026   Prob (F-statistic):           2.73e-25
Time:                        07:06:05   Log-Likelihood:                -8827.5
No. Observations:                5347   AIC:                         1.766e+04
Df Residuals:                    5343   BIC:                         1.769e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            1.1232      0.031  

## A2. Robust standard errors.

In [14]:
# OLS with Stata-equivalent heteroskedasticity-robust SE (HC1)
model2 = smf.ols(
    formula='ldurat ~ highearn * afchnge',
    data=df
).fit(cov_type='HC1', use_t=True)

print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                 ldurat   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     38.40
Date:                Mon, 26 Oct 2026   Prob (F-statistic):           1.52e-24
Time:                        07:06:05   Log-Likelihood:                -8827.5
No. Observations:                5347   AIC:                         1.766e+04
Df Residuals:                    5343   BIC:                         1.769e+04
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            1.1232      0.030  

## A3. Difference-in-differences with additional controls.

In [15]:
#check categorical
for col in ['indust', 'injtype', 'injdes']:
    print(f"\n=== {col} ===")
    print("Unique values:", df[col].unique())
    print("NaN count:", df[col].isna().sum())
    print("Dtype:", df[col].dtype)
    print("Value counts:")
    print(df[col].value_counts(dropna=False).sort_index())


=== indust ===
Unique values: [3. 1. 2.]
NaN count: 0
Dtype: float64
Value counts:
indust
1.0    1379
2.0     742
3.0    3226
Name: count, dtype: int64

=== injtype ===
Unique values: [1 2 3 4 5 6 8 7]
NaN count: 0
Dtype: int8
Value counts:
injtype
1     209
2      93
3    1576
4     587
5    1422
6    1245
7      45
8     170
Name: count, dtype: int64

=== injdes ===
Unique values: [1010 1404 1032 1940 1425 1110 1207 1510 1528 1910 1140 1340 1728 1107
 1437 1232 1540 1713 1410 1840 1710 1040 1928 1007 1013 2128 2152 2149
 2052 2010 2234 2049 2352 2552 2228 2216 3128 3416 3210 3540 3110 3152
 3413 3728 3228 3628 3510 3528 3613 3328 3337 3504 3640 3010 3310 3149
 3649 3140 3610 3302 3304 3449 3713 3428 3340 3740 3410 3137 3704 3440
 3452 3252 3725 3249 3237 3652 3225 3004 3647 3602 3516 3530 3116 3136
 3437 3447 3536 3513 3643 3355 3352 3049 3743 4440 4428 4534 4646 4634
 4110 4334 4849 4528 4834 4410 4452 4346 4152 4010 4552 4852 4036 4352
 4652 4613 4128 4837 4052 4846 4643 4434 4628

In [16]:
# OLS with add variables SE = HC1
model3 = smf.ols(
    formula='''
        ldurat ~ highearn * afchnge
                + lage
                + male
                + married
                + hosp
                + C(indust)
                + C(injtype)
                + C(injdes)
    ''',
    data=df
).fit(cov_type='HC1', use_t=True)

print("=== Model 3: Robust SE (HC1) with Controls ===")
print(model3.summary())


/tmp/ipykernel_1823/3294602934.py:14: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit(cov_type='HC1', use_t=True)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 371, but rank is 257
  res = self.wald_test(


=== Model 3: Robust SE (HC1) with Controls ===
                            OLS Regression Results                            
Dep. Variable:                 ldurat   R-squared:                       0.299
Model:                            OLS   Adj. R-squared:                  0.248
Method:                 Least Squares   F-statistic:                     40.72
Date:                Mon, 26 Oct 2026   Prob (F-statistic):               0.00
Time:                        07:06:06   Log-Likelihood:                -7935.8
No. Observations:                5347   AIC:                         1.660e+04
Df Residuals:                    4982   BIC:                         1.900e+04
Df Model:                         364                                         
Covariance Type:                  HC1                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------

## create table

In [17]:
#general setting
#model
models = [model1, model2, model3]

# Variable labels
labels = {
    'ldurat': 'Log duration of benefits (weeks)',
    'highearn': 'High earner',
    'afchnge': 'After benefit increase',
    'highearn:afchnge': r'High earner $\times$ After',
    'lage': 'Log age',
    'male': 'Male',
    'married': 'Married',
    'hosp': 'Hospitalized',
    'C(indust)': 'Industry fixed effects',
    'C(injtype)': 'Injury type fixed effects',
    'C(injdes)': 'Injury description fixed effects',
    'Intercept': 'Constant'
}

# DID coefficients and intercept
coef_vars = [
    'highearn',
    'afchnge',
    'highearn:afchnge',
    'Intercept'
]


# Additional control coefficients and standard errors
control_vars = ['lage', 'male', 'married', 'hosp']

# Fixed effects indicators
fe_vars = ['C(indust)', 'C(injtype)', 'C(injdes)']


In [18]:
# Create table
rows = []

# DID coefficients and standard errors
for var in coef_vars:
    rows.append(
        [labels[var]] + [f'{m.params[var]:.3f}' for m in models]
    )
    rows.append(
        [''] + [f'({m.bse[var]:.3f})' for m in models]
    )

for var in control_vars:
    rows.append(
        [labels[var]] + [
            f'{m.params[var]:.3f}' if var in m.params.index else ''
            for m in models
        ]
    )
    rows.append(
        [''] + [
            f'({m.bse[var]:.3f})' if var in m.params.index else ''
            for m in models
        ]
    )

for var in fe_vars:
    rows.append(
        [labels[var]] + [
            r'$\bigcirc$'
            if any(
                name.startswith(f'{var}[')
                for name in m.model.exog_names
            )
            else r'$\times$'
            for m in models
        ]
    )

# Standard error types
rows.append([
    'Standard errors',
    *[
        'Conventional' if m.cov_type == 'nonrobust'
        else 'HC1 robust'
        for m in models
    ]
])

# Sample sizes
rows.append([
    'Observations',
    *[f'{int(m.nobs):,}' for m in models]
])

# Construct regression table
table = pd.DataFrame(
    rows,
    columns=['Variable', '(1)', '(2)', '(3)']
)

# Export to LaTeX string
latex_table = table.to_latex(
    index=False,
    escape=False,
    column_format='lccc',
    caption='OLS Estimates of Log Duration of Benefits (Weeks)',
    label='tab:did_results'
)

print(latex_table)

\begin{table}
\caption{OLS Estimates of Log Duration of Benefits (Weeks)}
\label{tab:did_results}
\begin{tabular}{lccc}
\toprule
Variable & (1) & (2) & (3) \\
\midrule
High earner & 0.233 & 0.233 & 0.095 \\
 & (0.049) & (0.048) & (0.048) \\
After benefit increase & 0.014 & 0.014 & 0.012 \\
 & (0.045) & (0.045) & (0.042) \\
High earner $\times$ After & 0.229 & 0.229 & 0.164 \\
 & (0.070) & (0.070) & (0.064) \\
Constant & 1.123 & 1.123 & -0.265 \\
 & (0.031) & (0.030) & (0.354) \\
Log age &  &  & 0.163 \\
 &  &  & (0.049) \\
Male &  &  & -0.058 \\
 &  &  & (0.043) \\
Married &  &  & 0.055 \\
 &  &  & (0.037) \\
Hospitalized &  &  & 1.065 \\
 &  &  & (0.041) \\
Industry fixed effects & $\times$ & $\times$ & $\bigcirc$ \\
Injury type fixed effects & $\times$ & $\times$ & $\bigcirc$ \\
Injury description fixed effects & $\times$ & $\times$ & $\bigcirc$ \\
Standard errors & Conventional & HC1 robust & HC1 robust \\
Observations & 5,347 & 5,347 & 5,347 \\
\bottomrule
\end{tabular}
\end{table}